# Projeto 2ª VA: comparação de algoritmos de classificação

Base usada: Sleep Health and Lifestyle (a mesma da Entrega 1, já tratada).

O que queremos: descobrir se a pessoa não tem distúrbio do sono, se tem insônia ou se tem apneia.

O que vamos fazer neste notebook:
1. testar 6 algoritmos: Árvore de Decisão, KNN, Naive Bayes, Regressão Logística, SVM e MLP
2. avaliar cada um com validação cruzada de 10 folds (estratificada)
3. testar várias combinações de parâmetros com o GridSearchCV
4. mostrar a média e o desvio padrão dos resultados nos 10 folds.

In [41]:
# Instala as bibliotecas (só precisa rodar uma vez)
%pip install -q pandas numpy matplotlib scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    make_scorer, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay
)

# Desliga os avisos (warnings) para a saída ficar mais limpa
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

SEED = 42        # semente fixa, para o resultado ser sempre o mesmo
N_FOLDS = 10     # quantidade de folds da validação cruzada

# No arquivo tratado as classes viraram números: 0 = None, 1 = Insomnia, 2 = Sleep Apnea
NOMES_CLASSES = ["None", "Insomnia", "Sleep Apnea"]

# Pasta onde vamos salvar as tabelas e os gráficos
os.makedirs("resultados", exist_ok=True)

## 1. Carregando os dados

Aqui usamos a base que já foi tratada na Entrega 1 (`dados/sleep_preprocessado.csv`). A base original continua em `dados/sleep_original.csv`.

Na Entrega 1 ja havia sido feito a retirada da coluna de ID, juntado rótulos repetidos, separado a pressão arterial em duas colunas e transformado os textos em números. A base não tem valores faltando.

In [43]:
df = pd.read_csv("dados/sleep_preprocessado.csv")
alvo = "Sleep Disorder"   # coluna que queremos prever

print("Linhas e colunas:", df.shape)
print("\nQuantidade de pessoas em cada classe:")
print(df[alvo].value_counts().sort_index().rename(index=dict(enumerate(NOMES_CLASSES))))

Linhas e colunas: (374, 20)

Quantidade de pessoas em cada classe:
Sleep Disorder
None           219
Insomnia        77
Sleep Apnea     78
Name: count, dtype: int64


### Linhas repetidas

Na Entrega 1 vimos que 242 das 374 linhas são repetidas (são cópias de outras linhas).

Isso é um problema na validação cruzada: uma cópia pode ir para o treino e outra igualzinha para o teste. Aí o modelo "já viu a resposta" e o resultado parece melhor do que é.

Como o enunciado não fala nada sobre isso, rodamos tudo com as duas versões da base:
- completa: as 374 linhas
- sem_duplicatas: só as 132 linhas diferentes (cada classe ainda tem mais de 10 linhas, então ainda dá para fazer 10 folds).

In [44]:
# drop_duplicates() apaga as linhas repetidas e deixa só uma de cada
df_unico = df.drop_duplicates().reset_index(drop=True)
df_unico.to_csv("dados/sleep_preprocessado_sem_duplicatas.csv", index=False)

# Guardamos as duas versões num dicionário para rodar o mesmo código nas duas
bases = {"completa": df, "sem_duplicatas": df_unico}

for nome, d in bases.items():
    print(f"{nome}: {d.shape[0]} linhas | por classe:", d[alvo].value_counts().sort_index().tolist())

completa: 374 linhas | por classe: [219, 77, 78]
sem_duplicatas: 132 linhas | por classe: [73, 29, 30]


## 2. Como vamos avaliar

- Validação cruzada: a base é dividida em 10 partes (folds). Em cada rodada, 9 partes treinam e 1 parte testa. Isso se repete 10 vezes, até cada parte ter sido teste uma vez.
- Estratificada: cada parte fica com a mesma proporção de classes da base inteira.
- Os 10 folds são os mesmos para todos os algoritmos, assim a comparação é justa.
- Métricas: acurácia, precisão, revocação (recall) e F1. Usamos a média "macro", que dá o mesmo peso para as 3 classes. Sem isso, a classe None (que é a maior) dominaria o resultado.
- No final mostramos a média e o desvio padrão dos 10 folds.

In [45]:
# Divide os dados em 10 partes, mantendo a proporção das classes
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Métricas que vamos calcular em cada fold
# zero_division=0 evita erro se o modelo não prever nenhuma vez uma classe
METRICAS = {
    "accuracy": "accuracy",
    "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
}

# Só para conferir: quantas pessoas de cada classe caíram em cada fold de teste
X_c, y_c = df.drop(columns=alvo), df[alvo]
contagem = pd.DataFrame(
    [np.bincount(y_c.iloc[teste], minlength=3) for _, teste in skf.split(X_c, y_c)],
    columns=NOMES_CLASSES,
    index=[f"fold {i}" for i in range(1, N_FOLDS + 1)]
)
contagem

,None,Insomnia,Sleep Apnea
fold 1,22,8,8
fold 2,22,8,8
fold 3,22,8,8
fold 4,22,8,8
fold 5,22,7,8
fold 6,22,7,8
fold 7,22,7,8
fold 8,22,8,7
fold 9,22,8,7
fold 10,21,8,8


## 3. Algoritmos e parâmetros

O enunciado pede pelo menos 3 combinações de parâmetros por algoritmo. Aqui testamos de 5 a 12.

| Algoritmo | Parâmetros que variamos | Combinações |
|---|---|---|
| Árvore de Decisão | criterion, max_depth, min_samples_leaf | 12 |
| KNN | n_neighbors, weights | 10 |
| Naive Bayes (Gaussiano) | var_smoothing | 5 |
| Regressão Logística | C | 5 |
| SVM | kernel (linear ou rbf), C, gamma | 12 |
| MLP | hidden_layer_sizes, alpha | 6 |

Sobre a escala: alguns atributos têm números grandes (passos por dia, na casa dos milhares) e outros pequenos (estresse, de 3 a 8). Para o KNN, a SVM, a Regressão Logística e o MLP isso atrapalha, então colocamos um StandardScaler antes do modelo. Ele fica dentro de um Pipeline, assim a média e o desvio usados na padronização são calculados só com o treino de cada fold.

In [46]:
def com_escala(modelo, escalar=True):
    # Monta o Pipeline: primeiro padroniza (se escalar=True), depois roda o modelo
    etapas = [("scaler", StandardScaler())] if escalar else []
    return Pipeline(etapas + [("clf", modelo)])


# Para cada algoritmo: (modelo, parâmetros que vamos testar)
# O "clf__" na frente do nome é porque o modelo está dentro do Pipeline com o nome "clf"
experimentos = {
    # A árvore não precisa padronizar, ela não usa distância
    "Árvore de Decisão": (
        com_escala(DecisionTreeClassifier(random_state=SEED), escalar=False),
        {
            "clf__criterion": ["gini", "entropy"],
            "clf__max_depth": [3, 5, None],       # None = sem limite de profundidade
            "clf__min_samples_leaf": [1, 5],
        },
    ),
    # O KNN usa distância, então a escala faz diferença
    "KNN": (
        com_escala(KNeighborsClassifier()),
        {"clf__n_neighbors": [3, 5, 7, 11, 15], "clf__weights": ["uniform", "distance"]},
    ),
    # No Naive Bayes também padronizamos, para o var_smoothing valer igual para todos os atributos
    "Naive Bayes": (
        com_escala(GaussianNB()),
        {"clf__var_smoothing": [1e-9, 1e-6, 1e-3, 1e-2, 1e-1]},
    ),
    "Regressão Logística": (
        com_escala(LogisticRegression(max_iter=5000)),   # max_iter alto para dar tempo de convergir
        {"clf__C": [0.01, 0.1, 1, 10, 100]},
    ),
    # Na SVM, o gamma só existe no kernel rbf, por isso são duas grades
    "SVM": (
        com_escala(SVC()),
        [
            {"clf__kernel": ["linear"], "clf__C": [0.1, 1, 10]},
            {"clf__kernel": ["rbf"], "clf__C": [0.1, 1, 10], "clf__gamma": [0.01, 0.1, 1]},
        ],
    ),
    "MLP": (
        com_escala(MLPClassifier(max_iter=2000, random_state=SEED)),
        {"clf__hidden_layer_sizes": [(16,), (32,), (32, 16)], "clf__alpha": [1e-4, 1e-2]},
    ),
}

# Conta quantas combinações cada algoritmo vai testar
for nome, (_, grade) in experimentos.items():
    grades = grade if isinstance(grade, list) else [grade]
    n_comb = sum(np.prod([len(v) for v in g.values()]) for g in grades)
    print(f"{nome}: {n_comb} combinações")

Árvore de Decisão: 12 combinações
KNN: 10 combinações
Naive Bayes: 5 combinações
Regressão Logística: 5 combinações
SVM: 12 combinações
MLP: 6 combinações


## 4. Rodando o GridSearchCV

O GridSearchCV testa todas as combinações de parâmetros, cada uma nos 10 folds. Guardamos o resultado de cada fold, e não só a média, porque depois usamos esses valores nos gráficos e no teste estatístico.

A coluna `melhor` marca a combinação com maior F1 médio de cada algoritmo.

In [40]:
def rodar_grid(X, y):
    """Roda o GridSearchCV de cada algoritmo e devolve uma tabela com o resultado de cada fold."""
    por_fold = []     # aqui vão as linhas da tabela final
    melhores = {}     # aqui guardamos o melhor modelo de cada algoritmo

    for nome, (modelo, grade) in experimentos.items():
        # refit="f1_macro" o melhor modelo é escolhido pelo F1
        gs = GridSearchCV(modelo, grade, scoring=METRICAS, refit="f1_macro", cv=skf, n_jobs=-1)
        gs.fit(X, y)
        melhores[nome] = gs.best_estimator_
        res = gs.cv_results_

        # res guarda o resultado de cada combinação em cada fold (split0, split1, ...)
        for i, params in enumerate(res["params"]):
            comb = ", ".join(f"{k.replace('clf__', '')}={v}" for k, v in params.items())
            for fold in range(N_FOLDS):
                linha = {"algoritmo": nome, "combinacao": comb,
                         "melhor": i == gs.best_index_, "fold": fold + 1}
                for m in METRICAS:
                    linha[m] = res[f"split{fold}_test_{m}"][i]
                por_fold.append(linha)

    return pd.DataFrame(por_fold), melhores


resultados = {}          # resultado por fold de cada base
melhores_modelos = {}    # melhor modelo de cada algoritmo em cada base

for nome_base, dados in bases.items():
    X, y = dados.drop(columns=alvo), dados[alvo]
    resultados[nome_base], melhores_modelos[nome_base] = rodar_grid(X, y)
    resultados[nome_base].to_csv(f"resultados/{nome_base}_metricas_por_fold.csv", index=False)
    print(f"{nome_base}: terminou")

completa: terminou
sem_duplicatas: terminou


## 5. Resultado de cada combinação

Cada valor está como média ± desvio padrão dos 10 folds. A coluna `melhor` mostra "sim" na combinação escolhida de cada algoritmo.

In [48]:
def resumir(por_fold):
    # Junta os 10 folds de cada combinação e calcula a média e o desvio padrão
    agg = por_fold.groupby(["algoritmo", "combinacao", "melhor"], sort=False)[list(METRICAS)].agg(["mean", "std"])
    agg.columns = [f"{m}_{est}" for m, est in agg.columns]   # ex.: f1_macro_mean, f1_macro_std
    return agg.reset_index()


def formatar(tab):
    # Só deixa a tabela mais fácil de ler: "0.880 ± 0.073"
    out = tab[["algoritmo", "combinacao"]].copy()
    out["melhor"] = np.where(tab["melhor"], "sim", "")
    for m in METRICAS:
        out[m] = tab[f"{m}_mean"].map("{:.3f}".format) + " ± " + tab[f"{m}_std"].map("{:.3f}".format)
    return out


tabelas = {nome_base: resumir(pf) for nome_base, pf in resultados.items()}

for nome_base, tab in tabelas.items():
    tab.to_csv(f"resultados/{nome_base}_todas_combinacoes.csv", index=False)

### 5.1 Base completa (374 linhas)

In [49]:
for nome in experimentos:
    print(nome)
    display(formatar(tabelas["completa"].query("algoritmo == @nome")).drop(columns="algoritmo").reset_index(drop=True))

Árvore de Decisão


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,"criterion=gini, max_depth=3, min_samples_leaf=1",,0.895 ± 0.050,0.882 ± 0.063,0.862 ± 0.069,0.861 ± 0.069
1,"criterion=gini, max_depth=3, min_samples_leaf=5",,0.893 ± 0.044,0.884 ± 0.051,0.858 ± 0.061,0.861 ± 0.057
2,"criterion=gini, max_depth=5, min_samples_leaf=1",,0.898 ± 0.052,0.884 ± 0.071,0.867 ± 0.075,0.867 ± 0.076
3,"criterion=gini, max_depth=5, min_samples_leaf=5",,0.893 ± 0.040,0.880 ± 0.049,0.858 ± 0.060,0.862 ± 0.054
4,"criterion=gini, max_depth=None, min_samples_le...",,0.898 ± 0.036,0.886 ± 0.056,0.870 ± 0.045,0.871 ± 0.048
5,"criterion=gini, max_depth=None, min_samples_le...",,0.893 ± 0.040,0.882 ± 0.050,0.858 ± 0.060,0.861 ± 0.055
6,"criterion=entropy, max_depth=3, min_samples_le...",,0.890 ± 0.056,0.880 ± 0.064,0.854 ± 0.082,0.856 ± 0.076
7,"criterion=entropy, max_depth=3, min_samples_le...",,0.887 ± 0.050,0.884 ± 0.051,0.850 ± 0.075,0.856 ± 0.064
8,"criterion=entropy, max_depth=5, min_samples_le...",sim,0.904 ± 0.044,0.892 ± 0.059,0.876 ± 0.064,0.876 ± 0.063
9,"criterion=entropy, max_depth=5, min_samples_le...",,0.890 ± 0.043,0.876 ± 0.054,0.853 ± 0.065,0.858 ± 0.060


KNN


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,"n_neighbors=3, weights=uniform",,0.863 ± 0.047,0.845 ± 0.057,0.831 ± 0.081,0.830 ± 0.071
1,"n_neighbors=3, weights=distance",,0.887 ± 0.042,0.869 ± 0.058,0.872 ± 0.062,0.865 ± 0.060
2,"n_neighbors=5, weights=uniform",,0.882 ± 0.053,0.881 ± 0.065,0.844 ± 0.079,0.855 ± 0.071
3,"n_neighbors=5, weights=distance",,0.903 ± 0.053,0.893 ± 0.072,0.881 ± 0.066,0.882 ± 0.069
4,"n_neighbors=7, weights=uniform",,0.885 ± 0.058,0.878 ± 0.075,0.848 ± 0.084,0.855 ± 0.080
5,"n_neighbors=7, weights=distance",,0.903 ± 0.053,0.893 ± 0.072,0.881 ± 0.066,0.882 ± 0.069
6,"n_neighbors=11, weights=uniform",,0.869 ± 0.054,0.867 ± 0.064,0.818 ± 0.079,0.829 ± 0.075
7,"n_neighbors=11, weights=distance",,0.901 ± 0.050,0.890 ± 0.068,0.877 ± 0.062,0.878 ± 0.064
8,"n_neighbors=15, weights=uniform",,0.863 ± 0.062,0.869 ± 0.067,0.819 ± 0.080,0.828 ± 0.081
9,"n_neighbors=15, weights=distance",sim,0.903 ± 0.053,0.894 ± 0.068,0.881 ± 0.065,0.882 ± 0.067


Naive Bayes


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,var_smoothing=1e-09,,0.874 ± 0.053,0.855 ± 0.068,0.855 ± 0.068,0.848 ± 0.068
1,var_smoothing=1e-06,,0.885 ± 0.050,0.868 ± 0.063,0.864 ± 0.063,0.859 ± 0.064
2,var_smoothing=0.001,sim,0.887 ± 0.053,0.872 ± 0.070,0.869 ± 0.066,0.864 ± 0.068
3,var_smoothing=0.01,,0.879 ± 0.047,0.865 ± 0.065,0.857 ± 0.063,0.852 ± 0.062
4,var_smoothing=0.1,,0.882 ± 0.052,0.868 ± 0.069,0.858 ± 0.065,0.855 ± 0.066


... (2 outputs left out)

SVM


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,"C=0.1, kernel=linear",,0.895 ± 0.059,0.889 ± 0.077,0.868 ± 0.072,0.871 ± 0.074
1,"C=1, kernel=linear",,0.906 ± 0.037,0.898 ± 0.048,0.880 ± 0.045,0.882 ± 0.045
2,"C=10, kernel=linear",,0.906 ± 0.046,0.892 ± 0.071,0.877 ± 0.070,0.880 ± 0.070
3,"C=0.1, gamma=0.01, kernel=rbf",,0.826 ± 0.044,0.873 ± 0.055,0.740 ± 0.077,0.760 ± 0.073
4,"C=0.1, gamma=0.1, kernel=rbf",,0.858 ± 0.060,0.889 ± 0.059,0.801 ± 0.090,0.826 ± 0.081
5,"C=0.1, gamma=1, kernel=rbf",,0.852 ± 0.062,0.888 ± 0.055,0.787 ± 0.092,0.817 ± 0.083
6,"C=1, gamma=0.01, kernel=rbf",,0.893 ± 0.058,0.891 ± 0.076,0.862 ± 0.078,0.865 ± 0.077
7,"C=1, gamma=0.1, kernel=rbf",,0.906 ± 0.059,0.893 ± 0.080,0.882 ± 0.080,0.883 ± 0.080
8,"C=1, gamma=1, kernel=rbf",,0.906 ± 0.041,0.904 ± 0.057,0.874 ± 0.063,0.883 ± 0.059
9,"C=10, gamma=0.01, kernel=rbf",,0.903 ± 0.046,0.890 ± 0.068,0.878 ± 0.063,0.879 ± 0.064


MLP


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,"alpha=0.0001, hidden_layer_sizes=(16,)",,0.903 ± 0.056,0.890 ± 0.073,0.881 ± 0.071,0.880 ± 0.073
1,"alpha=0.0001, hidden_layer_sizes=(32,)",,0.906 ± 0.057,0.892 ± 0.078,0.885 ± 0.073,0.884 ± 0.075
2,"alpha=0.0001, hidden_layer_sizes=(32, 16)",sim,0.909 ± 0.054,0.895 ± 0.074,0.887 ± 0.073,0.887 ± 0.074
3,"alpha=0.01, hidden_layer_sizes=(16,)",,0.903 ± 0.056,0.890 ± 0.073,0.881 ± 0.071,0.880 ± 0.073
4,"alpha=0.01, hidden_layer_sizes=(32,)",,0.906 ± 0.057,0.892 ± 0.078,0.885 ± 0.073,0.884 ± 0.075
5,"alpha=0.01, hidden_layer_sizes=(32, 16)",,0.909 ± 0.054,0.895 ± 0.074,0.887 ± 0.073,0.887 ± 0.074


### 5.2 Base sem duplicatas (132 linhas)

In [50]:
for nome in experimentos:
    print(nome)
    display(formatar(tabelas["sem_duplicatas"].query("algoritmo == @nome")).drop(columns="algoritmo").reset_index(drop=True))

Árvore de Decisão


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,"criterion=gini, max_depth=3, min_samples_leaf=1",,0.727 ± 0.130,0.656 ± 0.203,0.665 ± 0.135,0.638 ± 0.160
1,"criterion=gini, max_depth=3, min_samples_leaf=5",sim,0.751 ± 0.134,0.675 ± 0.215,0.686 ± 0.141,0.664 ± 0.171
2,"criterion=gini, max_depth=5, min_samples_leaf=1",,0.675 ± 0.093,0.603 ± 0.176,0.609 ± 0.109,0.580 ± 0.124
3,"criterion=gini, max_depth=5, min_samples_leaf=5",,0.726 ± 0.111,0.671 ± 0.190,0.653 ± 0.111,0.638 ± 0.137
4,"criterion=gini, max_depth=None, min_samples_le...",,0.538 ± 0.040,0.387 ± 0.104,0.423 ± 0.066,0.391 ± 0.077
5,"criterion=gini, max_depth=None, min_samples_le...",,0.726 ± 0.111,0.675 ± 0.172,0.646 ± 0.115,0.636 ± 0.132
6,"criterion=entropy, max_depth=3, min_samples_le...",,0.721 ± 0.141,0.670 ± 0.207,0.668 ± 0.139,0.644 ± 0.179
7,"criterion=entropy, max_depth=3, min_samples_le...",,0.729 ± 0.132,0.670 ± 0.207,0.673 ± 0.134,0.648 ± 0.174
8,"criterion=entropy, max_depth=5, min_samples_le...",,0.652 ± 0.099,0.566 ± 0.163,0.575 ± 0.109,0.549 ± 0.126
9,"criterion=entropy, max_depth=5, min_samples_le...",,0.727 ± 0.135,0.656 ± 0.209,0.646 ± 0.143,0.632 ± 0.171


KNN


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,"n_neighbors=3, weights=uniform",,0.659 ± 0.154,0.606 ± 0.264,0.559 ± 0.169,0.552 ± 0.199
1,"n_neighbors=3, weights=distance",,0.569 ± 0.111,0.453 ± 0.150,0.479 ± 0.119,0.449 ± 0.121
2,"n_neighbors=5, weights=uniform",,0.689 ± 0.156,0.646 ± 0.240,0.604 ± 0.159,0.600 ± 0.185
3,"n_neighbors=5, weights=distance",,0.584 ± 0.130,0.489 ± 0.177,0.501 ± 0.138,0.479 ± 0.150
4,"n_neighbors=7, weights=uniform",,0.704 ± 0.163,0.636 ± 0.246,0.626 ± 0.171,0.613 ± 0.200
5,"n_neighbors=7, weights=distance",,0.584 ± 0.130,0.495 ± 0.169,0.501 ± 0.138,0.480 ± 0.147
6,"n_neighbors=11, weights=uniform",,0.712 ± 0.165,0.656 ± 0.248,0.637 ± 0.174,0.627 ± 0.199
7,"n_neighbors=11, weights=distance",,0.591 ± 0.127,0.500 ± 0.165,0.512 ± 0.136,0.487 ± 0.143
8,"n_neighbors=15, weights=uniform",sim,0.720 ± 0.149,0.663 ± 0.243,0.642 ± 0.165,0.635 ± 0.194
9,"n_neighbors=15, weights=distance",,0.591 ± 0.127,0.494 ± 0.173,0.512 ± 0.136,0.485 ± 0.146


Naive Bayes


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,var_smoothing=1e-09,,0.491 ± 0.180,0.505 ± 0.218,0.505 ± 0.143,0.432 ± 0.170
1,var_smoothing=1e-06,,0.521 ± 0.160,0.499 ± 0.156,0.510 ± 0.122,0.459 ± 0.149
2,var_smoothing=0.001,,0.681 ± 0.177,0.634 ± 0.204,0.651 ± 0.178,0.623 ± 0.190
3,var_smoothing=0.01,,0.726 ± 0.171,0.672 ± 0.219,0.679 ± 0.176,0.659 ± 0.195
4,var_smoothing=0.1,sim,0.734 ± 0.168,0.680 ± 0.214,0.684 ± 0.174,0.666 ± 0.190


... (2 outputs left out)

SVM


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,"C=0.1, kernel=linear",,0.742 ± 0.166,0.670 ± 0.235,0.681 ± 0.174,0.660 ± 0.202
1,"C=1, kernel=linear",,0.743 ± 0.146,0.697 ± 0.223,0.682 ± 0.154,0.660 ± 0.184
2,"C=10, kernel=linear",sim,0.729 ± 0.152,0.700 ± 0.210,0.666 ± 0.157,0.661 ± 0.180
3,"C=0.1, gamma=0.01, kernel=rbf",,0.553 ± 0.026,0.184 ± 0.009,0.333 ± 0.000,0.237 ± 0.007
4,"C=0.1, gamma=0.1, kernel=rbf",,0.553 ± 0.026,0.184 ± 0.009,0.333 ± 0.000,0.237 ± 0.007
5,"C=0.1, gamma=1, kernel=rbf",,0.553 ± 0.026,0.184 ± 0.009,0.333 ± 0.000,0.237 ± 0.007
6,"C=1, gamma=0.01, kernel=rbf",,0.727 ± 0.140,0.668 ± 0.226,0.653 ± 0.154,0.640 ± 0.180
7,"C=1, gamma=0.1, kernel=rbf",,0.720 ± 0.153,0.633 ± 0.217,0.648 ± 0.162,0.626 ± 0.183
8,"C=1, gamma=1, kernel=rbf",,0.698 ± 0.130,0.642 ± 0.239,0.602 ± 0.145,0.598 ± 0.178
9,"C=10, gamma=0.01, kernel=rbf",,0.720 ± 0.152,0.620 ± 0.218,0.648 ± 0.162,0.619 ± 0.186


MLP


,combinacao,melhor,accuracy,precision_macro,recall_macro,f1_macro
0,"alpha=0.0001, hidden_layer_sizes=(16,)",sim,0.735 ± 0.164,0.707 ± 0.217,0.684 ± 0.165,0.668 ± 0.184
1,"alpha=0.0001, hidden_layer_sizes=(32,)",,0.697 ± 0.151,0.606 ± 0.203,0.629 ± 0.164,0.602 ± 0.180
2,"alpha=0.0001, hidden_layer_sizes=(32, 16)",,0.561 ± 0.132,0.471 ± 0.177,0.482 ± 0.130,0.466 ± 0.147
3,"alpha=0.01, hidden_layer_sizes=(16,)",,0.735 ± 0.164,0.707 ± 0.217,0.684 ± 0.165,0.668 ± 0.184
4,"alpha=0.01, hidden_layer_sizes=(32,)",,0.697 ± 0.151,0.606 ± 0.203,0.629 ± 0.164,0.602 ± 0.180
5,"alpha=0.01, hidden_layer_sizes=(32, 16)",,0.576 ± 0.128,0.498 ± 0.172,0.504 ± 0.124,0.489 ± 0.142


## 6. Comparando os algoritmos

Aqui pegamos só a melhor combinação de cada algoritmo. Também colocamos a média geral dos 6, como o enunciado pede.

Dica para ler: se a diferença entre dois algoritmos é menor que o desvio padrão, não dá para dizer que um é melhor. Pode ser só sorte na divisão dos folds.

In [11]:
def tabela_melhores(tab):
    melhores = tab[tab["melhor"]].reset_index(drop=True)
    out = formatar(melhores).drop(columns=["combinacao", "melhor"])
    # Última linha: média geral dos 6 algoritmos
    media_geral = {"algoritmo": "Média geral"}
    for m in METRICAS:
        media_geral[m] = f"{melhores[m + '_mean'].mean():.3f}"
    return pd.concat([out, pd.DataFrame([media_geral])], ignore_index=True)


for nome_base, tab in tabelas.items():
    final = tabela_melhores(tab)
    final.to_csv(f"resultados/{nome_base}_melhores_por_algoritmo.csv", index=False)
    print(f"Base {nome_base}")
    display(final)

Base completa


,algoritmo,accuracy,precision_macro,recall_macro,f1_macro
0,Árvore de Decisão,0.904 ± 0.044,0.892 ± 0.059,0.876 ± 0.064,0.876 ± 0.063
1,KNN,0.903 ± 0.053,0.894 ± 0.068,0.881 ± 0.065,0.882 ± 0.067
2,Naive Bayes,0.887 ± 0.053,0.872 ± 0.070,0.869 ± 0.066,0.864 ± 0.068
3,Regressão Logística,0.906 ± 0.045,0.892 ± 0.066,0.880 ± 0.062,0.881 ± 0.063
4,SVM,0.909 ± 0.054,0.895 ± 0.074,0.887 ± 0.073,0.887 ± 0.074
5,MLP,0.909 ± 0.054,0.895 ± 0.074,0.887 ± 0.073,0.887 ± 0.074
6,Média geral,0.903,0.890,0.880,0.880


Base sem_duplicatas


,algoritmo,accuracy,precision_macro,recall_macro,f1_macro
0,Árvore de Decisão,0.751 ± 0.134,0.675 ± 0.215,0.686 ± 0.141,0.664 ± 0.171
1,KNN,0.720 ± 0.149,0.663 ± 0.243,0.642 ± 0.165,0.635 ± 0.194
2,Naive Bayes,0.734 ± 0.168,0.680 ± 0.214,0.684 ± 0.174,0.666 ± 0.190
3,Regressão Logística,0.742 ± 0.175,0.707 ± 0.253,0.682 ± 0.196,0.665 ± 0.221
4,SVM,0.729 ± 0.152,0.700 ± 0.210,0.666 ± 0.157,0.661 ± 0.180
5,MLP,0.735 ± 0.164,0.707 ± 0.217,0.684 ± 0.165,0.668 ± 0.184
6,Média geral,0.735,0.689,0.674,0.660


In [12]:
# Quais parâmetros ganharam em cada algoritmo
for nome_base, tab in tabelas.items():
    print(f"Base {nome_base}")
    display(tab[tab["melhor"]][["algoritmo", "combinacao"]].reset_index(drop=True))

Base completa


,algoritmo,combinacao
0,Árvore de Decisão,"criterion=entropy, max_depth=5, min_samples_le..."
1,KNN,"n_neighbors=15, weights=distance"
2,Naive Bayes,var_smoothing=0.001
3,Regressão Logística,C=100
4,SVM,"C=10, gamma=0.1, kernel=rbf"
5,MLP,"alpha=0.0001, hidden_layer_sizes=(32, 16)"


Base sem_duplicatas


,algoritmo,combinacao
0,Árvore de Decisão,"criterion=gini, max_depth=3, min_samples_leaf=5"
1,KNN,"n_neighbors=15, weights=uniform"
2,Naive Bayes,var_smoothing=0.1
3,Regressão Logística,C=1
4,SVM,"C=10, kernel=linear"
5,MLP,"alpha=0.0001, hidden_layer_sizes=(16,)"
